<a href="https://colab.research.google.com/github/Aniket-034/APS-LAB-/blob/main/Lab_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MATRIX CHAIN MULTIPLICATION Dynamic Programming with table filling.
Problem Description:
Find minimum number of scalar multiplications needed to multiply a chain of matrices..

#Theory
Cost of multiplying two matrices:
If A is p×q and B is q×r, cost = p × q × r

Dynamic Programming idea:
Break problem into smaller subproblems.
Store results in a table to avoid recomputation.

m[i][j] = minimum cost to multiply Ai to Aj

## Algorithm
1. Take array p[] of size n
2. Create table m[n][n]
3. Set m[i][i] = 0
4. Fill table diagonally
5. For each chain length L:
    Compute m[i][j]
    Try all possible k between i and j
6. Choose minimum cost
7. Return m[1][n-1]

In [8]:
import sys

def matrix_chain_multiplication(p):
    n = len(p)

    # Create DP table for costs
    m = [[0 for _ in range(n)] for _ in range(n)]
    # Create table for optimal split points
    s = [[0 for _ in range(n)] for _ in range(n)]

    # L is chain length
    for L in range(2, n): # L from 2 to n-1
        for i in range(1, n - L + 1): # i from 1 to n-L
            j = i + L - 1 # j from L to n-1
            m[i][j] = sys.maxsize

            for k in range(i, j): # k from i to j-1
                cost = (m[i][k] +
                        m[k+1][j] +
                        p[i-1] * p[k] * p[j])

                if cost < m[i][j]:
                    m[i][j] = cost
                    s[i][j] = k # Store the split point

    return m, s

In [6]:
def print_optimal_parenthesization(s, i, j):
    if i == j:
        return f"A{i}"
    else:
        return f"({print_optimal_parenthesization(s, i, s[i][j])}{print_optimal_parenthesization(s, s[i][j] + 1, j)})"

def calculate_naive_multiplications(p):
    # Calculates multiplications for a left-to-right parenthesization
    # (A1A2)A3...An
    cost = 0
    if len(p) <= 2: # No multiplications needed for 0 or 1 matrix
        return 0

    # First multiplication (A0A1)
    current_p = [p[0], p[1]]
    cost += p[0] * p[1] * p[2]

    for i in range(2, len(p) - 1):
        # (current_result)Ai+1
        current_p = [current_p[0], p[i]]
        cost += current_p[0] * current_p[1] * p[i+1]

    return cost

In [3]:
# Input dimensions
p = [10, 30, 5, 60]

table = matrix_chain_multiplication(p)

print("Minimum multiplication cost:", table[1][len(p)-1])

print("\nDP Table:")
for row in table[1:]:
    print(row[1:])

Minimum multiplication cost: 4500

DP Table:
[0, 1500, 4500]
[0, 0, 9000]
[0, 0, 0]


# Exercise
1. Given p = [5, 10, 3, 12, 5, 50, 6]
2. Compute minimum cost
3. Print optimal parenthesization
4. Compare number of multiplications saved

In [10]:
# Input dimensions for the exercise
p_exercise = [5, 10, 3, 12, 5, 50, 6]

m_table, s_table = matrix_chain_multiplication(p_exercise)

min_cost = m_table[1][len(p_exercise) - 1]
print(f"Minimum multiplication cost for p_exercise: {min_cost}")

print("\nDP Table (costs):")
for row in m_table[1:]:
    print(row[1:])

print("\nDP Table (split points):")
for row in s_table[1:]:
    print(row[1:])

optimal_paren = print_optimal_parenthesization(s_table, 1, len(p_exercise) - 1)
print(f"\nOptimal Parenthesization: {optimal_paren}")

# Calculate multiplications for naive left-to-right order
# Note: This is an approximation/simplified version for comparison.
# For (A1A2)A3...An, A1 is p[0]xp[1], A2 is p[1]xp[2], etc.
# The cost for A_i * A_{i+1} is p[i-1] * p[i] * p[i+1]
# For naive multiplication, it's (A0(A1(...(An-1An)...)))
# or, if we consider it (A0A1)A2...: first product is A0A1, then (A0A1)A2 etc.
# Let's consider the matrices M0, M1, M2, M3, M4, M5
# (M0M1)M2 => cost = p[0]*p[1]*p[2] + p[0]*p[2]*p[3]
# ((M0M1)M2)M3 => cost = p[0]*p[1]*p[2] + p[0]*p[2]*p[3] + p[0]*p[3]*p[4]

# Calculate naive multiplications cost
# This assumes matrices are A_0, A_1, ..., A_{n-1} where A_i is p[i] x p[i+1]
# For p=[d0,d1,d2,...,dn], matrices are M0(d0xd1), M1(d1xd2), ..., Mn-1(dn-1xdn)
# (M0 M1 ... Mn-1)

def calculate_left_to_right_cost(p_dims):
    if len(p_dims) <= 2:
        return 0

    cost = 0
    current_rows = p_dims[0]
    current_cols = p_dims[1]

    # First multiplication: M0 * M1. Dimensions: (p[0] x p[1]) * (p[1] x p[2])
    cost += current_rows * current_cols * p_dims[2]
    current_cols = p_dims[2] # Resulting matrix has dimensions (p[0] x p[2])

    # Subsequent multiplications: (Result) * M_i
    # Result is (p[0] x current_cols), M_i is (current_cols x p_dims[i+1])
    for i in range(3, len(p_dims)):
        cost += current_rows * current_cols * p_dims[i]
        current_cols = p_dims[i] # Resulting matrix has dimensions (p[0] x p[i])
    return cost

naive_cost = calculate_left_to_right_cost(p_exercise)
print(f"\nCost with naive left-to-right multiplication: {naive_cost}")

saved_multiplications = naive_cost - min_cost
print(f"Number of multiplications saved: {saved_multiplications}")

Minimum multiplication cost for p_exercise: 2010

DP Table (costs):
[0, 150, 330, 405, 1655, 2010]
[0, 0, 360, 330, 2430, 1950]
[0, 0, 0, 180, 930, 1770]
[0, 0, 0, 0, 3000, 1860]
[0, 0, 0, 0, 0, 1500]
[0, 0, 0, 0, 0, 0]

DP Table (split points):
[0, 1, 2, 2, 4, 2]
[0, 0, 2, 2, 2, 2]
[0, 0, 0, 3, 4, 4]
[0, 0, 0, 0, 4, 4]
[0, 0, 0, 0, 0, 5]
[0, 0, 0, 0, 0, 0]

Optimal Parenthesization: ((A1A2)((A3A4)(A5A6)))

Cost with naive left-to-right multiplication: 3380
Number of multiplications saved: 1370
